In [2]:
# 1. Install required ecosystem dependencies quietly
!pip install pymongo dnspython tensorflow pillow opencv-python-headless -q

import os
import cv2
import base64
import numpy as np
import tensorflow as tf
from PIL import Image
from datetime import datetime, timezone
from pymongo import MongoClient
from google.colab import files, userdata

In [3]:
print("🚀 Environment initialized. Starting Drowsiness Inference Engine...")

# ==========================================
# 2. DATABASE CONNECTIVITY LAYER
# ==========================================
try:
    # Safely fetch your connection string from Colab Secrets
    MONGO_URI = userdata.get('MONGO_URI')
    client = MongoClient(MONGO_URI, serverSelectionTimeoutMS=3000)
    client.admin.command('ping') # Test cluster handshake

    # Target database and the specific "users_data" collection requested
    db = client['drowsiness_detection_db']
    user_collection = db['users_data']
    print("✅ Successfully connected to MongoDB Cloud Cluster!")
except Exception as db_err:
    print(f"❌ Database connection failed: {db_err}")
    print("💡 Please verify your 'MONGO_URI' secret is set correctly in the Key sidebar tab.")
    user_collection = None


🚀 Environment initialized. Starting Drowsiness Inference Engine...
✅ Successfully connected to MongoDB Cloud Cluster!


In [5]:
# ==========================================
# 3. MACHINE LEARNING MODEL INITIALIZATION
# ==========================================
MODEL_PATH = 'ultimate_drowsiness_system.h5'

if not os.path.exists(MODEL_PATH):
    raise FileNotFoundError(f"❌ Critical Error: '{MODEL_PATH}' was not found in the file manager sidebar. Please upload your model first.")

print("🧠 Loading Core Neural Network Weights (this may take a few seconds)...")
prediction_model = tf.keras.models.load_model(MODEL_PATH)
face_cascade = cv2.CascadeClassifier(cv2.data.haarcascades + 'haarcascade_frontalface_default.xml')
print("✅ AI Model & Face Detectors loaded successfully!")


🧠 Loading Core Neural Network Weights (this may take a few seconds)...
✅ AI Model & Face Detectors loaded successfully!


In [7]:




# ==========================================
# 4. INTERACTIVE FILE UPLOADER
# ==========================================
print("\n👇 CLICK BELOW TO UPLOAD DRIVER PHOTO (.jpg, .jpeg, .png):")
uploaded = files.upload()

if uploaded:
    # Grab the filename and raw data bytes
    filename = list(uploaded.keys())[0]
    file_bytes = uploaded[filename]

    # Convert file bytes into an OpenCV image matrix format
    np_arr = np.frombuffer(file_bytes, np.uint8)
    cv_img = cv2.imdecode(np_arr, cv2.IMREAD_COLOR)

    # Initialize metric baseline parameters
    pipeline_status = "SUCCESS"
    status = "REJECTED_BY_PIPELINE"
    confidence = 0.0
    recommendation = "N/A"
    error_details = None

    # Convert image bytes to a Base64 string layout for MongoDB storage
    encoded_image_string = base64.b64encode(file_bytes).decode('utf-8')

    try:
        # Pre-validation: Confirm face presence to guarantee accuracy tracking
        gray_img = cv2.cvtColor(cv_img, cv2.COLOR_BGR2GRAY)
        faces = face_cascade.detectMultiScale(gray_img, scaleFactor=1.1, minNeighbors=5, minSize=(30, 30))

        if len(faces) == 0:
            raise ValueError("No human face detected. System rejected the photo to preserve prediction accuracy.")

        # Format the matrix to match your model input criteria (128x128 resolution)
        img_resized = cv2.resize(cv_img, (128, 128))
        img_array = tf.keras.preprocessing.image.img_to_array(img_resized)
        img_array = np.expand_dims(img_array, axis=0)

        # Run Deep Learning Classification Inference
        prediction = prediction_model.predict(img_array, verbose=0)
        score = float(prediction[0][0])

        # Establish binary outcome labels
        if score < 0.5:
            status = "DROWSY"
            confidence = float((1 - score) * 100)
            recommendation = "⚠️ ALERT: Pull over safely and take a break immediately!"
        else:
            status = "ALERT / NON-DROWSY"
            confidence = float(score * 100)
            recommendation = "🚗 SAFE: System detects an active, alert driver."

        # ==========================================
        # 5. PRINT RESULTS DIRECTLY IN COLAB
        # ==========================================
        print("\n=============================================")
        print("📊 INFERENCE ENGINE DIAGNOSTICS")
        print("=============================================")
        print(f"🔹 Prediction Output: {status}")
        print(f"🔹 Model Confidence : {confidence:.2f}%")
        print(f"🔹 Recommendation   : {recommendation}")
        print("=============================================")

    except Exception as pipeline_error:
        pipeline_status = "FAILED"
        status = "EXCEPTION_HANDLED"
        error_details = str(pipeline_error)
        print(f"\n❌ System Pipeline Exception: {error_details}")

    finally:
        # ==========================================
        # 6. STREAM TELEMETRY DATA TO MONGODB
        # ==========================================
        if user_collection is not None:
            # Build structured analytics profile document
            telemetry_document = {
                "timestamp": datetime.now(timezone.utc),
                "filename": filename,
                "pipeline_execution": pipeline_status,
                "status": status,
                "model_confidence": round(confidence, 2) if confidence > 0 else None,
                "recommendation": recommendation,
                "user_image": encoded_image_string,
                "internal_logs": {
                    "has_error": (pipeline_status == "FAILED"),
                    "error_message": error_details
                }
            }
            try:
                write_receipt = user_collection.insert_one(telemetry_document)
                print(f"\n🚀 Telemetry document successfully synchronized under collection 'users_data'!")
                print(f"🆔 Document MongoDB ID: {write_receipt.inserted_id}")
            except Exception as db_write_err:
                print(f"\n❌ Failed writing log data payload to Atlas Cloud Cluster: {db_write_err}")
        else:
            print("\nℹ️ System Warning: Telemetry data skipped. Establish your MONGO_URI token connection to log info.")
else:
    print("❌ Cancelled: No file selected or uploaded.")


👇 CLICK BELOW TO UPLOAD DRIVER PHOTO (.jpg, .jpeg, .png):


Saving pic.jpeg to pic.jpeg

📊 INFERENCE ENGINE DIAGNOSTICS
🔹 Prediction Output: ALERT / NON-DROWSY
🔹 Model Confidence : 80.85%
🔹 Recommendation   : 🚗 SAFE: System detects an active, alert driver.

🚀 Telemetry document successfully synchronized under collection 'users_data'!
🆔 Document MongoDB ID: 6a1c2348b354d491c08e7da4
